# 03. RAG 인덱스 구축

두 종류의 검색 가능한 데이터를 구축한다:
1. 매뉴얼/SOP 인덱스 - 'data/docs/*.md' 마크다운 문서
2. 고장 이력 인덱스 - 01번 노트북에서 생성한 'failure_history.csv'

임베딩 모델: 'paraphrase-multilingual-MiniLM-L12-v2' (384차원)

검색기: HybridRetriever = FAISS semantic + IDF 가중 키워드 점수 합산

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.rag.indexer import FaissIndexer, Chunk, load_docs
from src.rag.retriever import HybridRetriever, load_retriever
from src.data.synthetic_history import build_history_df, to_text_records

INDEX_DIR = PROJECT_ROOT / 'models_artifacts' / 'rag'
INDEX_DIR.mkdir(parents=True, exist_ok=True)
print('imports OK')

imports OK


## 1. 매뉴얼/SOP 인덱싱

'data/docs/'의 마크다운을 헤더 단위 청크로 분할 후 임베딩.

In [2]:
manual_chunks = load_docs(PROJECT_ROOT / 'data' / 'docs')
print(f'문서 청크: {len(manual_chunks)}개')
for c in manual_chunks[:3]:
    print(f'\n[{c.id}] {c.metadata}')
    print(c.text[:200], '...')

문서 청크: 36개

[MANUAL-001_apu_overview#0] {'source': 'MANUAL-001_apu_overview.md', 'title': 'MANUAL-001: APU 압축기 시스템 개요', 'type': 'manual'}
[MANUAL-001: APU 압축기 시스템 개요]
**문서번호**: MANUAL-001
**적용 설비**: APU-01, APU-02, APU-03 ...

[MANUAL-001_apu_overview#1] {'source': 'MANUAL-001_apu_overview.md', 'title': 'MANUAL-001: APU 압축기 시스템 개요', 'type': 'manual'}
[MANUAL-001: APU 압축기 시스템 개요 > 1. 시스템 구성]
APU(Air Production Unit)는 메트로 차량에 압축 공기를 공급하는 단위 장치이다.
- 모터 + 압축기 본체 + 토출 밸브(DV) + 저장조(Reservoirs) + 쿨러 + 제어부 ...

[MANUAL-001_apu_overview#2] {'source': 'MANUAL-001_apu_overview.md', 'title': 'MANUAL-001: APU 압축기 시스템 개요', 'type': 'manual'}
[MANUAL-001: APU 압축기 시스템 개요 > 2. 센서 구성 > 2.1 아날로그 센서 (7종)]
| 센서 | 단위 | 정상 범위 | 설명 |
|------|------|----------|------|
| TP2 | bar | 7.0 ~ 9.5 | 압축기 토출부 압력 |
| TP3 | bar | 8.0 ~ 9.5 | 패널부 압력 (소비 후 잔압)  ...


In [3]:
manual_indexer = FaissIndexer(device='cpu')
manual_indexer.build(manual_chunks)
manual_indexer.save(INDEX_DIR / 'manual')
print(f'saved → {INDEX_DIR / "manual"}')

d:\Work\study_full_data\log_anomaly_dectetion_pipeline\log-anomaly-detection\.claude\worktrees\epic-turing-62c439\llm_agent_phm\src\rag\indexer.py:118: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

saved → d:\Work\study_full_data\log_anomaly_dectetion_pipeline\log-anomaly-detection\.claude\worktrees\epic-turing-62c439\llm_agent_phm\models_artifacts\rag\manual


## 2. 고장 이력 인덱싱

In [4]:
history_df = build_history_df()
records = to_text_records(history_df)
history_chunks = [Chunk(id=r['id'], text=r['text'], metadata={**r['metadata'], 'type': 'history'}) for r in records]
print(f'이력 청크: {len(history_chunks)}개')
print('\n예시:')
print(history_chunks[0].text)

이력 청크: 8개

예시:
[사례 INC-2020-0317] 2020-03-17 10:20 | 설비: APU-01
증상: 정기 점검 중 H1 압력 노이즈 증가 발견, 운전엔 영향 없음
진단: 센서 드리프트 (Sensor Drift)
근본 원인: 압력 센서 노후 (사용 시간 25,000h 초과)
조치: 센서 신품 교체 및 영점 보정
다운타임: 25분 / 작업자: 김OO


In [5]:
history_indexer = FaissIndexer(device='cpu')
history_indexer.build(history_chunks)
history_indexer.save(INDEX_DIR / 'history')
print(f'saved → {INDEX_DIR / "history"}')

d:\Work\study_full_data\log_anomaly_dectetion_pipeline\log-anomaly-detection\.claude\worktrees\epic-turing-62c439\llm_agent_phm\src\rag\indexer.py:118: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

saved → d:\Work\study_full_data\log_anomaly_dectetion_pipeline\log-anomaly-detection\.claude\worktrees\epic-turing-62c439\llm_agent_phm\models_artifacts\rag\history


## 3. HybridRetriever 검색 테스트

In [6]:
manual_retriever = load_retriever(INDEX_DIR / 'manual', alpha=0.7)
history_retriever = load_retriever(INDEX_DIR / 'history', alpha=0.7)

queries = [
    ('manual',  '압력이 회복되지 않을 때 어떻게 점검하나요?'),
    ('manual',  'oil temperature 80도 넘었을 때'),
    ('history', 'TP2 압력 이상 + 모터 전류 상승'),
    ('history', 'APU-03 오일 누설'),
]
for kind, q in queries:
    r = manual_retriever if kind == 'manual' else history_retriever
    print(f'\n[{kind}] Q: {q}')
    for h in r.search(q, k=2):
        print(f'  ({h.score:.3f}) {h.chunk.id}')
        print(f'    {h.chunk.text[:120]}...')

d:\Work\study_full_data\log_anomaly_dectetion_pipeline\log-anomaly-detection\.claude\worktrees\epic-turing-62c439\llm_agent_phm\src\rag\indexer.py:118: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()



[manual] Q: 압력이 회복되지 않을 때 어떻게 점검하나요?
  (0.524) SOP-002_air_leak_response#1
    [SOP-002: 공기 누설(Air Leak) 대응 절차 > 1. Air Leak 판정 기준]
다음 중 2개 이상 충족 시 Air Leak 의심:
- TP2 압력이 7.0 bar 미만으로 회복되지 않음
- 압축기 무...
  (0.428) SOP-001_compressor_inspection#5
    [SOP-001: APU 압축기 일상 점검 절차 > 3. 이상 발견 시 조치]
1. 즉시 운전 일지 기록 (시간/현상/조치)
2. 안전한 경우 5분 모니터링 후 자가 회복 여부 확인
3. 회복되지 않으면 SOP-00...

[manual] Q: oil temperature 80도 넘었을 때
  (0.663) SOP-001_compressor_inspection#3
    [SOP-001: APU 압축기 일상 점검 절차 > 2. 일상 점검 항목 > 2.2 온도 점검]
- Oil_temperature: 60 ~ 75°C 정상, 80°C 초과 시 즉시 부하 감소
- 주변 환기구 청결 여부...
  (0.475) SOP-003_oil_temperature#2
    [SOP-003: 오일 온도/누설 이상 대응 절차 > 1. 판정 기준 > 1.2 오일 누설 의심]
- Oil_level 디지털 신호 LOW 알람
- 크랭크케이스 외부 오일 흔적
- 오일 보충 주기가 평소의 1/2 이...

[history] Q: TP2 압력 이상 + 모터 전류 상승
  (0.434) INC-2020-0508
    [사례 INC-2020-0508] 2020-05-08 03:15 | 설비: APU-02
증상: 야간 무부하 운전 중 TP3 압력 7.2 → 5.8bar 저하 반복
진단: 공기 누설 (Air Leak)
근본 원인: R...
  (0.424) INC-2020-0701
    [사례 INC-2020-0701] 2020-07-01 11:45 | 

d:\Work\study_full_data\log_anomaly_dectetion_pipeline\log-anomaly-detection\.claude\worktrees\epic-turing-62c439\llm_agent_phm\src\rag\indexer.py:118: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


## 4. 메타데이터 필터 검색

In [7]:
hits = history_retriever.search('공기 누설', k=3, filter_metadata={'equipment_id': 'APU-02'})
for h in hits:
    print(f'({h.score:.3f}) {h.chunk.id} | {h.chunk.metadata}')
    print(f'  {h.chunk.text[:100]}...')

(0.628) INC-2020-0508 | {'equipment_id': 'APU-02', 'diagnosis': '공기 누설 (Air Leak)', 'date': '2020-05-08T03:15:00', 'type': 'history'}
  [사례 INC-2020-0508] 2020-05-08 03:15 | 설비: APU-02
증상: 야간 무부하 운전 중 TP3 압력 7.2 → 5.8bar 저하 반복
진단: 공기 누설...
(0.597) INC-2020-0701 | {'equipment_id': 'APU-02', 'diagnosis': '공기 누설 + 압축기 부하 증가', 'date': '2020-07-01T11:45:00', 'type': 'history'}
  [사례 INC-2020-0701] 2020-07-01 11:45 | 설비: APU-02
증상: Motor_current 피크가 정상 5.2A 대비 7.8A까지 상승, 압력 회복 시...
